In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col,expr,current_timestamp , current_date ,lit ,sum
from delta.tables import DeltaTable

class QuantityProcessor:
    def __init__(self):
        self.spark = spark
        self.checkpoint_location_quantity = '**'

    def read_transaction_data(self) -> DataFrame:
        return (
            self.spark.readStream.table("dev.spark_db.transaction")
        )


    def available_balance_update(self,txn_df):

        agg_txn_df = (
            txn_df.groupBy(col("account_number"))
                                .agg(sum(col("order_amt")).alias("order_amt"))
        )


        account = DeltaTable.forPath(self.spark,
                                     "s3://investment-streaming-bucket/investment-streaming/deltatables/Account/")


        (
            account.alias("a").merge(agg_txn_df.alias("ad"),
                                     expr("a.account_number = ad.account_number"))
                              .whenMatchedUpdate(
                                  set = {
                                            "available_quantity":col("ad.quantity")+ col("a.available_quantity"),
                                            "account_ver" : col("a.account_ver") + lit(1),
                                            "account_dml" : current_timestamp() 
                                         }
                              ).execute
         
         )
        
    def quantity_update(self,txn_df):

        txn_qty = (txn_df.groupBy(col("account_number"),col("asset_external_id"))
                        .agg(sum(col("quantity")).alias("quantity"))
        )


        quantity = DeltaTable.forPath(self.spark,
                                      "s3://investment-streaming-bucket/investment-streaming/deltatables/Quantity/")
        
        (
            quantity.alias("qty").merge(txn_qty.alias("sr"),
                                        expr("""
                                             qty.account_number = sr.account_number
                                             and
                                             qty.asset_external_id = sr.asset_external_id
                                             """))
                                .whenMatchedUpdate(
                                    set = {
                                        "available_quantity" : col("qty.available_quantity") + col("sr.quantity"),
                                        "quantity_ver" : col("qty.quantity_ver") + lit(1),
                                        "quantity_dml" : current_timestamp()
                                    }
                                )
                                .whenNotMatchedInsert(
                                    values = {
                                        "account_number":col("sr.account_number"),
                                        "asset_external_id" : col("sr.asset_external_id"),
                                        "available_quantity" : col("sr.quantity"),
                                        "quantity_ver":lit(1),
                                        "quantity_dml":current_timestamp()
                                    }
                                ).execute()
        )

    def quantity_update_dynamodb(self,txn_qty):
        
        txn_qty = (txn_qty.groupBy(col("account_number"),col("asset_external_id"))
                        .agg(sum(col("quantity")).alias("quantity"))
                        .selectExpr("account_number",
                                    "asset_external_id",
                                    "quantity",
                                    "1 as quantity_ver",
                                    "current_timestamp() as quantity_dml")
        )

        def process_quantity(rows):
            import boto3

            dynamodb = boto3.resource(
                                        "dynamodb",
                                        region_name = '**',
                                        aws_access_key_id = '**',
                                        aws_secret_access_key = '**'
                                        )
            quantity = dynamodb.Table("quantity")

            for row in rows:
                response = quantity.update_item(
                    Key = {
                        "account_number" : row['account_number'],
                        "asset_external_id" : row['asset_external_id']
                    },
                    UpdateExpression = """SET available_quantity = if_not_exists(available_quantity, :zero) +     :quantity,
                                        quantity_ver = if_not_exists(quantity_ver , :zero) + :quantity_ver,
                                        quantity_dml = :quantity_dml
                                        """,
                    ExpressionAttributeValues = {
                        ":zero" : 0,
                        ":quantity" : row['quantity'],
                        ":quantity_ver": row['quantity_ver'],
                        ":quantity_dml" : str(row['quantity_dml'])
                    },
                    ReturnValues = 'ALL_NEW'
                )


        txn_qty.foreachPartition(process_quantity)



    def quantity_process(self,batch_df,batch_id):
        self.available_balance_update(batch_df)
        self.quantity_update(batch_df)
        self.quantity_update_dynamodb(batch_df)


    def write_to_dynamo(self,txn_df):
        return (txn_df.writeStream.format("delta")
                          .queryName("quantity_query")
                          .foreachBatch(self.quantity_process)
                          .outputMode("update")
                          .option("checkpointLocation" , self.checkpoint_location_quantity)
                          .trigger(availableNow = True)
                          .start()
        )

    def main(self):

        txn_df = self.read_transaction_data()
        self.write_to_dynamo(txn_df)

if __name__ == '__main__':
    op = QuantityProcessor()
    op.main()



        